# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import sys
import json
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from openai import OpenAI
from dotenv import load_dotenv
sys.path.append(os.path.abspath('..'))


In [3]:
load_dotenv(dotenv_path='../.env')
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print("Credentials loaded.")


Credentials loaded.


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
chroma_client = chromadb.PersistentClient(path="games_chromadb")
openai_key = os.getenv("OPENAI_API_KEY")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small",
    api_base="https://openai.vocareum.com/v1/"
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 
    """
    res = collection.query(query_texts=[query], n_results=3)
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    results_list = []
    for i in range(len(docs)):
        m = metas[i] if i < len(metas) else {}
        results_list.append({
            "Platform": m.get("Platform", "Unknown"),
            "Name": m.get("Name", "Unknown"),
            "YearOfRelease": m.get("YearOfRelease", 0),
            "Description": m.get("Description", "")
        })
    return results_list

test_retrieval = retrieve_game("FIFA 21")
print(f"Retrieved {len(test_retrieval)} games.")


Retrieved 3 games.


#### Evaluate Retrieval Tool

In [5]:
from pydantic import BaseModel, Field

class EvaluationReport(BaseModel):
    useful: bool = Field(description="True if the retrieved documents are sufficient to answer the query, False otherwise")
    description: str = Field(description="Brief explanation of why the context is useful or not")

    def __getitem__(self, item):
        if item == "status":
            return "SUFFICIENT" if self.useful else "INSUFFICIENT"
        if item == "explanation":
            return self.description
        return getattr(self, item)

    def get(self, item, default=None):
        try:
            return self[item]
        except AttributeError:
            return default

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openai.vocareum.com/v1/"
)

def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
    # Based on the user's question and on the list of retrieved documents, 
    # it will analyze the usability of the documents to respond to that question. 
    # args: 
    # - question: original question from user
    # - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    context = "\n".join([f"- {d.get('Name')} ({d.get('Platform')}): {d.get('Description')}" for d in retrieved_docs])
    
    system_message = (
        "You are an expert game research evaluator. Your task is to evaluate if the retrieved documents "
        "are enough to respond to the query."
    )
    
    user_message = f"Query: {question}\n\nRetrieved Docs:\n{context}"
    
    try:
        response = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            response_format=EvaluationReport,
            temperature=0.0
        )
        return response.choices[0].message.parsed
    except Exception as e:
        system_message_json = system_message + " Reply in JSON format: {\"useful\": true or false, \"description\": \"explanation\"}"
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_message_json},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.0
            )
            raw_content = response.choices[0].message.content.strip()
            if raw_content.startswith("```"):
                lines = raw_content.splitlines()
                if len(lines) > 2:
                    raw_content = "\n".join(lines[1:-1])
            parsed = json.loads(raw_content)
            return EvaluationReport(useful=parsed.get("useful", False), description=parsed.get("description", ""))
        except Exception as ex:
            return EvaluationReport(useful=False, description=f"Evaluation failed: {e} -> {ex}")


#### Game Web Search Tool

In [6]:
tavily_key = os.getenv("TAVILY_API_KEY")
if tavily_key and tavily_key != "YOUR_KEY":
    tavily_client = TavilyClient(api_key=tavily_key)
else:
    tavily_client = None

def game_web_search(question: str) -> list:
    # Perform a web search using the Tavily API to find video game information.
    # args:
    # - question: a question about the game industry. 
    if not tavily_client:
        raise ValueError("Tavily API client is not configured.")
        
    try:
        res = tavily_client.search(query=question, max_results=3)
        results = res.get("results", [])
        return [{
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": r.get("content", "")
        } for r in results]
    except Exception as e:
        print(f"Tavily search failed: {e}")
        return []


### Agent

In [7]:
class UdaPlayAgent:
    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url="https://openai.vocareum.com/v1/"
        )
        self.sessions = {}
        self.system_prompt = (
            "You are UdaPlay, an expert AI Research Assistant for the video game industry.\n"
            "Your goal is to provide accurate, comprehensive, and well-structured reports/answers "
            "about video games, including details on developers, release dates, platforms, genres, and publishers.\n\n"
            "You have access to the following tools:\n"
            "1. `retrieve_game`: Search the local vector database for game details. Always call this first.\n"
            "2. `evaluate_retrieval`: Evaluate whether retrieved local database results contain sufficient information to answer the query.\n"
            "3. `game_web_search`: Perform a web search to find missing or up-to-date details.\n\n"
            "Instructions:\n"
            "- Always analyze the conversation history first. If the user's query is a follow-up question (e.g. 'What platform was that on?') and the information is already in the history, answer directly without calling tools.\n"
            "- Otherwise, you MUST call `retrieve_game` first to search the local database for game information.\n"
            "- Once you get the local results, you MUST call `evaluate_retrieval` to evaluate if they are sufficient to answer the question.\n"
            "- If `evaluate_retrieval` reports that local database results are INSUFFICIENT (useful is false), you MUST call `game_web_search` to search the web for the missing details.\n"
            "- If `evaluate_retrieval` reports that results are SUFFICIENT (useful is true), do NOT call `game_web_search`.\n"
            "- Always cite your sources clearly in your final response (e.g. 'Source: Local Database' or using the URL from web results).\n"
            "- Present your final output in a clean, natural, and readable format."
        )

    def query(self, question: str, session_id: str = "default") -> str:
        if session_id not in self.sessions:
            self.sessions[session_id] = []
            
        messages = [{"role": "system", "content": self.system_prompt}]
        for h in self.sessions[session_id]:
            messages.append(h)
        messages.append({"role": "user", "content": question})
        
        tools_schema = [
            {
                "type": "function",
                "function": {
                    "name": "retrieve_game",
                    "description": "Queries the ChromaDB vector database for video game information.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "query": {
                                "type": "string",
                                "description": "The search query (e.g. the name of the game)."
                            }
                        },
                        "required": ["query"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "evaluate_retrieval",
                    "description": "Evaluates if retrieved database results are sufficient to answer the user query.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "question": {
                                "type": "string",
                                "description": "The original user query or question."
                            },
                            "retrieved_docs": {
                                "type": "array",
                                "items": {"type": "object"},
                                "description": "The list of documents returned by retrieve_game."
                            }
                        },
                        "required": ["question", "retrieved_docs"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "game_web_search",
                    "description": "Performs a web search using Tavily to find missing or up-to-date video game details.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "question": {
                                "type": "string",
                                "description": "The web search query."
                            }
                        },
                        "required": ["question"]
                    }
                }
            }
        ]
        
        called_web_search = False
        loop_count = 0
        max_loops = 10
        
        while loop_count < max_loops:
            loop_count += 1
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools_schema,
                tool_choice="auto",
                temperature=0.0
            )
            
            msg = response.choices[0].message
            tool_calls = msg.tool_calls
            
            if not tool_calls:
                final_answer = msg.content or ""
                self.sessions[session_id].append({"role": "user", "content": question})
                self.sessions[session_id].append({"role": "assistant", "content": final_answer})
                
                if called_web_search:
                    try:
                        doc_id = f"web_{len(question) % 1000}_{os.urandom(2).hex()}"
                        new_doc = f"[Web] Info: {final_answer}"
                        collection.add(
                            ids=[doc_id],
                            documents=[new_doc],
                            metadatas=[{"Name": question, "Platform": "Web", "YearOfRelease": 2026, "Description": final_answer}]
                        )
                        print(f"Persisted web search findings to memory under ID: {doc_id}")
                    except Exception as e:
                        print(f"Memory persistence warning: {e}")
                
                return final_answer
                
            messages.append(msg)
            
            for tool_call in tool_calls:
                tool_id = tool_call.id
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                print(f"🤖 [Tool Call] Calling tool '{tool_name}' with arguments: {tool_args}")
                
                if tool_name == "retrieve_game":
                    results = retrieve_game(tool_args.get("query", question))
                    tool_result = json.dumps(results)
                    
                elif tool_name == "evaluate_retrieval":
                    q = tool_args.get("question", question)
                    docs = tool_args.get("retrieved_docs", [])
                    if isinstance(docs, str):
                        try:
                            docs = json.loads(docs)
                        except:
                            docs = []
                    report = evaluate_retrieval(q, docs)
                    print(f"Evaluation: {report.description}")
                    tool_result = json.dumps({
                        "useful": report.useful,
                        "description": report.description,
                        "status": report["status"],
                        "explanation": report["explanation"]
                    })
                    
                elif tool_name == "game_web_search":
                    called_web_search = True
                    q = tool_args.get("question", question)
                    results = game_web_search(q)
                    tool_result = json.dumps(results)
                    
                else:
                    tool_result = f"Error: Unknown tool {tool_name}"
                
                short_result = tool_result[:150] + "..." if len(tool_result) > 150 else tool_result
                print(f"🔍 [Tool Result] Tool '{tool_name}' returned: {short_result}")
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_id,
                    "name": tool_name,
                    "content": tool_result
                })
        
        return "Agent error: max loop limit exceeded."

    def invoke(self, question: str, session_id: str = "default") -> str:
        # Alias for query to match the invoke pattern.
        return self.query(question, session_id=session_id)

agent = UdaPlayAgent()
print("Agent initialized.")


Agent initialized.


In [8]:
# Run queries with session ids to demonstrate session identity and conversation state
session_pokemon = "session_pokemon"
session_mario = "session_mario"
session_mkx = "session_mkx"

print("--- Starting Session: Pokémon Gold/Silver ---")
q1 = "When was Pokémon Gold and Silver released?"
a1 = agent.query(q1, session_id=session_pokemon)
print(f"Answer:\n{a1}\n")

# Follow-up question in the same session to demonstrate conversation continuity
q1_followup = "What platform was that on?"
print(f"\nProcessing follow-up query: '{q1_followup}' in session '{session_pokemon}'")
a1_followup = agent.query(q1_followup, session_id=session_pokemon)
print(f"Answer:\n{a1_followup}\n")

print("--- Starting Session: Mario ---")
q2 = "Which one was the first 3D platformer Mario game?"
a2 = agent.query(q2, session_id=session_mario)
print(f"Answer:\n{a2}\n")

print("--- Starting Session: Mortal Kombat ---")
q3 = "Was Mortal Kombat X released for PlayStation 5?"
a3 = agent.query(q3, session_id=session_mkx)
print(f"Answer:\n{a3}\n")


--- Starting Session: Pokémon Gold/Silver ---


🤖 [Tool Call] Calling tool 'retrieve_game' with arguments: {'query': 'Pokémon Gold and Silver'}


🔍 [Tool Result] Tool 'retrieve_game' returned: [{"Platform": "Game Boy Color", "Name": "Pok\u00e9mon Gold and Silver", "YearOfRelease": 1999, "Description": "Second-generation Pok\u00e9mon games in...


🤖 [Tool Call] Calling tool 'evaluate_retrieval' with arguments: {'question': 'When was Pokémon Gold and Silver released?', 'retrieved_docs': [{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'GameCube', 'Name': 'Super Smash Bros. Melee', 'YearOfRelease': 2001, 'Description': 'A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.'}]}


Evaluation: The retrieved documents do not contain the release date for Pokémon Gold and Silver. They provide information about other Pokémon games but do not address the specific query.
🔍 [Tool Result] Tool 'evaluate_retrieval' returned: {"useful": false, "description": "The retrieved documents do not contain the release date for Pok\u00e9mon Gold and Silver. They provide information a...


...

🤖 [Tool Call] Calling tool 'evaluate_retrieval' with arguments: {'question': 'Was Mortal Kombat X released for PlayStation 5?', 'retrieved_docs': [{'Platform': 'Xbox Series X|S', 'Name': 'Halo Infinite', 'YearOfRelease': 2021, 'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting."}, {'Platform': 'GameCube', 'Name': 'Super Smash Bros. Melee', 'YearOfRelease': 2001, 'Description': 'A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.'}, {'Platform': 'PlayStation 4', 'Name': "Marvel's Spider-Man", 'YearOfRelease': 2018, 'Description': 'An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.'}]}


Evaluation: The retrieved documents do not contain any information related to Mortal Kombat X or its release on PlayStation 5. They focus on other games and platforms, making them irrelevant to the query.
🔍 [Tool Result] Tool 'evaluate_retrieval' returned: {"useful": false, "description": "The retrieved documents do not contain any information related to Mortal Kombat X or its release on PlayStation 5. T...


🤖 [Tool Call] Calling tool 'game_web_search' with arguments: {'question': 'Mortal Kombat X release for PlayStation 5'}


🔍 [Tool Result] Tool 'game_web_search' returned: [{"title": "PS5 PlayStation Plus Collection To Include Mortal Kombat X", "url": "https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collect...


Persisted web search findings to memory under ID: web_47_3f03
Answer:
Mortal Kombat X was not released as a standalone title for the PlayStation 5, but it is available through the PlayStation Plus Collection. This means that players who subscribe to PlayStation Plus on PS5 can download and play Mortal Kombat X, which was originally released for PlayStation 4, Xbox One, and PC in 2015. 

Additionally, the game is compatible with the PlayStation 5, allowing players to enjoy it on the next-generation console. 

For more details, you can check the following sources:
- [Mortal Kombat X on Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)
- [Mortal Kombat X on GameStop](https://www.gamestop.com/video-games/products/mortal-kombat-x/11003580.html)
- [Mortal Kombat X in the PlayStation Plus Collection](https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V)



### (Optional) Advanced

In [9]:
print("\n--- Demonstrating memory recall (no web search fallback needed) ---")
a3_memory = agent.query("Was Mortal Kombat X released for PlayStation 5?", session_id="session_recall")
print(f"Answer from Memory:\n{a3_memory}\n")



--- Demonstrating memory recall (no web search fallback needed) ---


🤖 [Tool Call] Calling tool 'retrieve_game' with arguments: {'query': 'Mortal Kombat X'}


🔍 [Tool Result] Tool 'retrieve_game' returned: [{"Platform": "Web", "Name": "Was Mortal Kombat X released for PlayStation 5?", "YearOfRelease": 2026, "Description": "Mortal Kombat X was not release...


🤖 [Tool Call] Calling tool 'evaluate_retrieval' with arguments: {'question': 'Was Mortal Kombat X released for PlayStation 5?', 'retrieved_docs': [{'Platform': 'Web', 'Name': 'Was Mortal Kombat X released for PlayStation 5?', 'YearOfRelease': 2026, 'Description': 'Mortal Kombat X was not released as a standalone title for the PlayStation 5, but it is available through the PlayStation Plus Collection. This means that players who subscribe to PlayStation Plus on PS5 can download and play Mortal Kombat X, which was originally released for PlayStation 4, Xbox One, and PC in 2015. \n\nAdditionally, the game is compatible with the PlayStation 5, allowing players to enjoy it on the next-generation console. \n\nFor more details, you can check the following sources:\n- [Mortal Kombat X on Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)\n- [Mortal Kombat X on GameStop](https://www.gamestop.com/video-games/products/mortal-kombat-x/11003580.html)\n- [Mortal Kombat X in the PlayStation Pl

Evaluation: The retrieved documents provide a clear answer to the query, stating that Mortal Kombat X was not released as a standalone title for PlayStation 5 but is available through the PlayStation Plus Collection. It also mentions compatibility with the PS5, which directly addresses the user's question.
🔍 [Tool Result] Tool 'evaluate_retrieval' returned: {"useful": true, "description": "The retrieved documents provide a clear answer to the query, stating that Mortal Kombat X was not released as a stand...


Answer from Memory:
Mortal Kombat X was not released as a standalone title for the PlayStation 5. However, it is available through the PlayStation Plus Collection, meaning that players who subscribe to PlayStation Plus on PS5 can download and play it. The game, originally released for PlayStation 4, Xbox One, and PC in 2015, is also compatible with the PlayStation 5, allowing players to enjoy it on the next-generation console.

For more details, you can check the following sources:
- [Mortal Kombat X on Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)
- [Mortal Kombat X on GameStop](https://www.gamestop.com/video-games/products/mortal-kombat-x/11003580.html)
- [Mortal Kombat X in the PlayStation Plus Collection](https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V)

